<a href="https://colab.research.google.com/github/chipojaya1/sawso-education-data-system/blob/main/DRC_School_Assessment_App_v0_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



<h1 align="center">
    SAWSO EDUCATIONAL APP DEVELOPMENT
</h1>
  
<h2 align="center">
    Project: DRC
</h2>

<h3 align="center">
    Developer: Chipo Jaya
</h3>


## **Project Description:**

The first milestone for the Python/Gradio app is:
```
One school → one assessment criterion → one saved submission.
```
That means proving the full basic workflow before expanding to all 11 criteria:
  1. Select Province → Level → School
  2. Auto-fill the school identifiers
  3. Enter assessor information
  4. Complete Criterion 1: Infrastructure
  5. Add the provisional score and evidence note
  6. Click Save Assessment
  7. Successfully write that record to a dataframe/CSV

So the first milestone is not “build the whole app.” It is to prove that one complete assessment can move from selection to saved data end-to-end.

<h3 align = "center">
    Milestone #1
</h3>

## Step 1 — Setting up the environment, installing required packages and some basic data exploration

Packages for app development:
```
pandas: to handle the school register and assessment data.
openpyxl: allows Python to read the Excel workbook.
gradio: creates the actual app interface.!pip install -q gradio pandas openpyxl

In [22]:
import gradio as gr
import pandas as pd

print("Gradio version:", gr.__version__)
print("Pandas version:", pd.__version__)
print("Environment ready.")

Gradio version: 6.29.0
Pandas version: 2.2.3
Environment ready.


## Step 2 Uploading the current DRC Kobo workbook
```
For now, we are not changing anything. We to confirm that it can see the workbook and its sheets.

In [23]:
# Uploading the current DRC Kobo workbook
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
# Load and inspect the data
file_path = "/content/drive/MyDrive/SAWSO/DRC/Kobo form/DRC_School_Assessment_Kobo_Starter_v3.3.xlsx"

import os
import pandas as pd

print("File exists:", os.path.exists(file_path))

excel_file = pd.ExcelFile(file_path)

print("\nSheets found:")
print(excel_file.sheet_names)

choices_df = pd.read_excel(file_path, sheet_name="choices")

print("\nChoices sheet shape:")
print(choices_df.shape)

print("\nFirst 10 rows:")
display(choices_df.head(10))

File exists: True

Sheets found:
['survey', 'choices', 'settings']

Choices sheet shape:
(718, 12)

First 10 rows:


,list_name,name,label::English (en),province,cona_id,level_key,secope_id,school_key,school_name,municipality,education_level,source_school_name_raw
0,assessor_role,education_officer,Education officer,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,assessor_role,school_manager,School manager / administrator,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,assessor_role,salvation_army_officer,Salvation Army officer,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,assessor_role,external_assessor,External assessor,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,assessor_role,other,Other,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,province,haut_katanga,Haut-Katanga,haut_katanga,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,province,haut_lomami,Haut-Lomami,haut_lomami,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,province,kasai_central,Kasai-Central,kasai_central,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,province,kinshasa,Kinshasa,kinshasa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,province,kongo_central,Kongo Central,kongo_central,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Identifying school choice list and level choice list

In [25]:
# See all choice lists and how many rows each contains
list_counts = (
    choices_df["list_name"]
    .value_counts(dropna=False)
    .rename_axis("list_name")
    .reset_index(name="row_count")
)

display(list_counts)

,list_name,row_count
0,school,615
1,reg_level,27
2,support_need,11
3,province,10
4,support_source,8
5,governance_participants,7
6,school_level,5
7,assessor_role,5
8,frequency,5
9,adequacy_4,4


In [26]:
# Filter the choices DataFrame to see the options for 'school' and 'reg_level'
school_choices = choices_df[choices_df['list_name'] == 'school']
level_choices = choices_df[choices_df['list_name'] == 'reg_level']

print(f"Number of schools listed: {len(school_choices)}")
print(f"Number of levels listed: {len(level_choices)}")

print("\nSample schools (First 10):")
display(school_choices[['name', 'label::English (en)']].head(10))

print("\nAll registration levels:")
display(level_choices[['name', 'label::English (en)']])

Number of schools listed: 615
Number of levels listed: 27

Sample schools (First 10):


,name,label::English (en)
42,hkp001,EP Imani
43,hkp002,EP Matondo
44,hkp003,EP2 Tujenge
45,hkp004,EP Wokuvu
46,hks005,Institut Imani
47,hks006,Institut Matondo
48,hks007,Institut Tujenge
49,hks008,Institut Wokovu
50,hlp001,EP Cisumba Mukulu
51,hlp002,EP Diambangisa



All registration levels:


,name,label::English (en)
15,primary,Primary
16,secondary,Secondary
17,primary,Primary
18,secondary,Secondary
19,nursery,Nursery
20,primary,Primary
21,secondary,Secondary
22,nursery,Nursery
23,primary,Primary
24,secondary,Secondary


In [27]:
# Calculate percentages and display the full list of choice counts
list_counts['percentage'] = (list_counts['row_count'] / list_counts['row_count'].sum() * 100).round(2)

print("Summary of Choice Lists (sorted by row count):")
display(list_counts)

# Quick analysis description of the most prominent lists
top_list = list_counts.iloc[0]
print(f"\nThe largest choice list is '{top_list['list_name']}' with {top_list['row_count']} items, making up {top_list['percentage']}% of all choices.")

Summary of Choice Lists (sorted by row count):


,list_name,row_count,percentage
0,school,615,85.65
1,reg_level,27,3.76
2,support_need,11,1.53
3,province,10,1.39
4,support_source,8,1.11
5,governance_participants,7,0.97
6,school_level,5,0.70
7,assessor_role,5,0.70
8,frequency,5,0.70
9,adequacy_4,4,0.56



The largest choice list is 'school' with 615 items, making up 85.65% of all choices.


In [28]:
# Show rows that actually contain school records
school_rows_preview = choices_df[
    choices_df["school_key"].notna()
].copy()

print("Rows with a school_key:", len(school_rows_preview))

display(
    school_rows_preview[
        [
            "list_name",
            "name",
            "label::English (en)",
            "province",
            "level_key",
            "school_key",
            "cona_id",
            "secope_id",
            "school_name",
            "municipality",
            "education_level"
        ]
    ].head(10)
)

Rows with a school_key: 615


,list_name,name,label::English (en),province,level_key,school_key,cona_id,secope_id,school_name,municipality,education_level
42,school,hkp001,EP Imani,haut_katanga,primary,hkp001,HKP001,7017460,EP Imani,"235, Av. Basilique / Kenya",Primary Education
43,school,hkp002,EP Matondo,haut_katanga,primary,hkp002,HKP002,7051719,EP Matondo,"16 & 18 Av, Mwadinguha / Lubumbasi",Primary Education
44,school,hkp003,EP2 Tujenge,haut_katanga,primary,hkp003,HKP003,7017468,EP2 Tujenge,Kalukuluku / Ruashi / Lubambashi,Primary Education
45,school,hkp004,EP Wokuvu,haut_katanga,primary,hkp004,HKP004,7017312,EP Wokuvu,"104, Av. Kabongo / Katuba/Lubumbashi",Primary Education
46,school,hks005,Institut Imani,haut_katanga,secondary,hks005,HKS005,not_recorded,Institut Imani,"235, Av. Basilique / Kenya",Secondary Education
47,school,hks006,Institut Matondo,haut_katanga,secondary,hks006,HKS006,not_recorded,Institut Matondo,NaN,Secondary Education
48,school,hks007,Institut Tujenge,haut_katanga,secondary,hks007,HKS007,7055141,Institut Tujenge,Kalukuluku / Ruashi / Lubambashi,Secondary Education
49,school,hks008,Institut Wokovu,haut_katanga,secondary,hks008,HKS008,7017309,Institut Wokovu,"104, Av. Kabongo / Katuba/Lubumbashi",Secondary Education
50,school,hlp001,EP Cisumba Mukulu,haut_lomami,primary,hlp001,HLP001,not_recorded,EP Cisumba Mukulu,Q. 53 / Kasongo-Nyembo / Kamina,Primary Education
51,school,hlp002,EP Diambangisa,haut_lomami,primary,hlp002,HLP002,7167173,EP Diambangisa,Nzowa / Kamina,Primary Education


## STEP 3: Create a clean school master for the Python app

In [29]:
# Keep only the 615 school records
schools_df = choices_df[
    choices_df["list_name"] == "school"
].copy()

# Create a province lookup so the app shows "Haut-Katanga"
# instead of the internal value "haut_katanga"
province_lookup = (
    choices_df[choices_df["list_name"] == "province"]
    [["name", "label::English (en)"]]
    .drop_duplicates()
    .rename(
        columns={
            "name": "province",
            "label::English (en)": "province_label"
        }
    )
)

# Add the readable province name to each school
schools_df = schools_df.merge(
    province_lookup,
    on="province",
    how="left"
)

# Keep only the fields that will matter to the app
schools_df = schools_df[
    [
        "school_key",
        "label::English (en)",
        "province",
        "province_label",
        "level_key",
        "cona_id",
        "secope_id",
        "municipality",
        "education_level",
        "school_name",
        "source_school_name_raw"
    ]
].copy()

# Rename the user-facing school label
schools_df = schools_df.rename(
    columns={
        "label::English (en)": "school_label"
    }
)

# Sort for easier inspection
schools_df = schools_df.sort_values(
    by=["province_label", "level_key", "school_label"]
).reset_index(drop=True)

# ---------------------------------------------------------
# QUALITY CHECKS
# ---------------------------------------------------------

print("School master shape:", schools_df.shape)
print("Number of unique schools:", schools_df["school_key"].nunique())
print("Number of provinces:", schools_df["province"].nunique())
print("Level keys found:", sorted(schools_df["level_key"].dropna().unique()))
print("Duplicate school keys:", schools_df["school_key"].duplicated().sum())

print("\nSchools per province:")
display(
    schools_df.groupby("province_label")
    .size()
    .reset_index(name="school_count")
)

print("\nClean school master preview:")
display(schools_df.head(10))

School master shape: (615, 11)
Number of unique schools: 615
Number of provinces: 10
Level keys found: ['nursery', 'primary', 'secondary']
Duplicate school keys: 0

Schools per province:


,province_label,school_count
0,Haut-Katanga,8
1,Haut-Lomami,11
2,Kasai-Central,41
3,Kinshasa,83
4,Kongo Central,185
5,Kwilu / Bandundu,29
6,Mongala,85
7,"North Kivu, South Kivu & Maniema",11
8,Tanganyika,6
9,Tshopo,156



Clean school master preview:


,school_key,school_label,province,province_label,level_key,cona_id,secope_id,municipality,education_level,school_name,source_school_name_raw
0,hkp001,EP Imani,haut_katanga,Haut-Katanga,primary,HKP001,7017460,"235, Av. Basilique / Kenya",Primary Education,EP Imani,EP Imani
1,hkp002,EP Matondo,haut_katanga,Haut-Katanga,primary,HKP002,7051719,"16 & 18 Av, Mwadinguha / Lubumbasi",Primary Education,EP Matondo,EP Matondo
2,hkp004,EP Wokuvu,haut_katanga,Haut-Katanga,primary,HKP004,7017312,"104, Av. Kabongo / Katuba/Lubumbashi",Primary Education,EP Wokuvu,EP Wokuvu
3,hkp003,EP2 Tujenge,haut_katanga,Haut-Katanga,primary,HKP003,7017468,Kalukuluku / Ruashi / Lubambashi,Primary Education,EP2 Tujenge,EP2 Tujenge
4,hks005,Institut Imani,haut_katanga,Haut-Katanga,secondary,HKS005,not_recorded,"235, Av. Basilique / Kenya",Secondary Education,Institut Imani,Institut Imani
5,hks006,Institut Matondo,haut_katanga,Haut-Katanga,secondary,HKS006,not_recorded,NaN,Secondary Education,Institut Matondo,Institut Matondo
6,hks007,Institut Tujenge,haut_katanga,Haut-Katanga,secondary,HKS007,7055141,Kalukuluku / Ruashi / Lubambashi,Secondary Education,Institut Tujenge,Institut Tujenge
7,hks008,Institut Wokovu,haut_katanga,Haut-Katanga,secondary,HKS008,7017309,"104, Av. Kabongo / Katuba/Lubumbashi",Secondary Education,Institut Wokovu,Institut Wokovu
8,hlp001,EP Cisumba Mukulu,haut_lomami,Haut-Lomami,primary,HLP001,not_recorded,Q. 53 / Kasongo-Nyembo / Kamina,Primary Education,EP Cisumba Mukulu,EP Cisumba Mukulu
9,hlp002,EP Diambangisa,haut_lomami,Haut-Lomami,primary,HLP002,7167173,Nzowa / Kamina,Primary Education,EP Diambangisa,EP Diambangisa


## STEP 4: Build the first Gradio school-selection screen
Now we'll build the first working app screen:

```
Province → Level → School → auto-filled school details
```

In [30]:
# Helper functions
def get_levels(province_label):
    """
    Return only the school levels that actually exist
    in the selected province.
    """
    if not province_label:
        return gr.Dropdown(
            choices=[],
            value=None
        )

    levels = (
        schools_df[
            schools_df["province_label"] == province_label
        ]["level_key"]
        .dropna()
        .unique()
        .tolist()
    )

    # Display labels for the user
    level_labels = {
        "nursery": "Nursery",
        "primary": "Primary",
        "secondary": "Secondary"
    }

    choices = [
        (level_labels.get(level, level.title()), level)
        for level in sorted(levels)
    ]

    return gr.Dropdown(
        choices=choices,
        value=None
    )


def get_schools(province_label, level_key):
    """
    Return only schools matching the selected
    province and level.
    """
    if not province_label or not level_key:
        return gr.Dropdown(
            choices=[],
            value=None
        )

    filtered = schools_df[
        (schools_df["province_label"] == province_label) &
        (schools_df["level_key"] == level_key)
    ].copy()

    filtered = filtered.sort_values("school_label")

    # User sees school_label
    # App stores school_key
    choices = list(
        zip(
            filtered["school_label"],
            filtered["school_key"]
        )
    )

    return gr.Dropdown(
        choices=choices,
        value=None
    )


def get_school_details(school_key):
    """
    Auto-fill school identifiers and other details
    after the school is selected.
    """
    if not school_key:
        return "", "", "", "", ""

    row = schools_df[
        schools_df["school_key"] == school_key
    ].iloc[0]

    cona = row["cona_id"]
    secope = row["secope_id"]
    municipality = row["municipality"]
    education_level = row["education_level"]

    # Make missing values user-friendly
    if pd.isna(cona):
        cona = "Not recorded"

    if pd.isna(secope) or secope == "not_recorded":
        secope = "Not recorded"

    if pd.isna(municipality):
        municipality = "Not recorded"

    if pd.isna(education_level):
        education_level = "Not recorded"

    return (
        row["school_key"],
        str(cona),
        str(secope),
        str(municipality),
        str(education_level)
    )


# ---------------------------------------------------------
# App interface
# ---------------------------------------------------------

province_choices = sorted(
    schools_df["province_label"]
    .dropna()
    .unique()
    .tolist()
)

with gr.Blocks(title="DRC School Assessment") as demo:

    gr.Markdown(
        """
        # DRC School Assessment
        ### School Identification

        Select the province, school level, and school.
        School identifiers will fill automatically.
        """
    )

    province_dropdown = gr.Dropdown(
        choices=province_choices,
        label="Province",
        value=None
    )

    level_dropdown = gr.Dropdown(
        choices=[],
        label="School Level",
        value=None
    )

    school_dropdown = gr.Dropdown(
        choices=[],
        label="School",
        value=None,
        filterable=True
    )

    gr.Markdown("### School Details")

    school_key_box = gr.Textbox(
        label="Internal School Key",
        interactive=False
    )

    cona_box = gr.Textbox(
        label="CONA ID",
        interactive=False
    )

    secope_box = gr.Textbox(
        label="SECOPE ID",
        interactive=False
    )

    municipality_box = gr.Textbox(
        label="Municipality / Territory / Locality",
        interactive=False
    )

    education_level_box = gr.Textbox(
        label="Recorded Education Level",
        interactive=False
    )

    # -----------------------------------------------------
    # Events
    # -----------------------------------------------------

    province_dropdown.change(
        fn=get_levels,
        inputs=province_dropdown,
        outputs=level_dropdown
    )

    level_dropdown.change(
        fn=get_schools,
        inputs=[
            province_dropdown,
            level_dropdown
        ],
        outputs=school_dropdown
    )

    school_dropdown.change(
        fn=get_school_details,
        inputs=school_dropdown,
        outputs=[
            school_key_box,
            cona_box,
            secope_box,
            municipality_box,
            education_level_box
        ]
    )


# Launch app
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5e64717b5ae1faa79c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://5e64717b5ae1faa79c.gradio.live


## STEP 5: Inspect the existing XLSForm survey structure
```
In the output, I want to identify exactly:
- the assessor fields;
- where Assessment Criterion 1 begins;
- the questions underneath Criterion 1;
- its evidence-note field;
- its 1–4 score field.


In [32]:
# Helper functions

survey_df = pd.read_excel(
    file_path,
    sheet_name="survey"
)

print("Survey sheet shape:", survey_df.shape)

print("\nSurvey columns:")
print(survey_df.columns.tolist())

# Show the first 80 rows with the fields most useful for app design
columns_to_show = [
    col for col in [
        "type",
        "name",
        "label::English (en)",
        "hint::English (en)",
        "required",
        "calculation",
        "relevant"
    ]
    if col in survey_df.columns
]

display(
    survey_df[columns_to_show]
    .head(80)
)

Survey sheet shape: (150, 12)

Survey columns:
['type', 'name', 'label::English (en)', 'hint::English (en)', 'required', 'default', 'constraint', 'constraint_message::English (en)', 'appearance', 'choice_filter', 'calculation', 'relevant']


,type,name,label::English (en),hint::English (en),required,calculation,relevant
0,start,start,NaN,NaN,NaN,NaN,NaN
1,end,end,NaN,NaN,NaN,NaN,NaN
2,today,today,NaN,NaN,NaN,NaN,NaN
3,username,username,NaN,NaN,NaN,NaN,NaN
4,deviceid,deviceid,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
75,end_repeat,NaN,NaN,NaN,NaN,NaN,NaN
76,select_one score_1_4,attendance_score,Student attendance and admissions: provisional...,"Use 1=very poor, 2=poor, 3=good, 4=very good. ...",True,NaN,NaN
77,text,attendance_evidence,Student attendance and admissions: evidence / ...,Briefly record the evidence supporting the score.,True,NaN,NaN
78,end_group,NaN,NaN,NaN,NaN,NaN,NaN


Notes: The survey loaded correctly. The 80-row preview skipped over some of the fields we need, so rather than guessing, we will extract exactly the assessor section and Criterion 1 from the v3.3 form.

### STEP 5B: Find assessor fields and Criterion 1 exactly
---

What I need from the output is just:
- the assessor-related rows
- the Criterion 1 block

Then we can build the first proper app section directly from the Kobo design instead of recreating it from memory.

In [33]:
# 1. Find anything related to the assessor
assessor_mask = (
    survey_df["name"].fillna("").str.contains("assessor", case=False) |
    survey_df["label::English (en)"].fillna("").str.contains("assessor", case=False)
)

print("ASSESSOR-RELATED ROWS")
display(
    survey_df.loc[
        assessor_mask,
        [
            "type",
            "name",
            "label::English (en)",
            "hint::English (en)",
            "required"
        ]
    ]
)

# ---------------------------------------------------------
# 2. Locate Criterion 1
# ---------------------------------------------------------

criterion1_rows = survey_df[
    (survey_df["name"] == "c1") |
    (
        survey_df["type"].astype(str).str.contains("begin_group", na=False) &
        survey_df["label::English (en)"].fillna("").str.startswith("1.")
    )
]

print("\nCRITERION 1 START")
display(
    criterion1_rows[
        [
            "type",
            "name",
            "label::English (en)"
        ]
    ]
)

# ---------------------------------------------------------
# 3. Show the full Criterion 1 block
# ---------------------------------------------------------

if len(criterion1_rows) > 0:

    start_index = criterion1_rows.index[0]

    # Find the next end_group after Criterion 1 starts
    following_rows = survey_df.loc[start_index:]

    end_candidates = following_rows[
        following_rows["type"] == "end_group"
    ]

    if len(end_candidates) > 0:
        end_index = end_candidates.index[0]
    else:
        end_index = start_index + 20

    print(f"\nCRITERION 1 ROWS: {start_index} to {end_index}")

    display(
        survey_df.loc[
            start_index:end_index,
            [
                "type",
                "name",
                "label::English (en)",
                "hint::English (en)",
                "required",
                "relevant"
            ]
        ]
    )

else:
    print("Criterion 1 group was not found.")

ASSESSOR-RELATED ROWS


,type,name,label::English (en),hint::English (en),required
9,text,assessor_name,Assessor name,NaN,True
10,select_one assessor_role,assessor_role,Assessor role,NaN,True
11,text,assessor_role_other,Other assessor role,NaN,True
12,begin_group,group_bi7gf72,Assessor Contact Details,NaN,False
44,select_one score_1_4,infra_score,Infrastructure: provisional assessor score,"Use 1=very poor, 2=poor, 3=good, 4=very good. ...",True
45,text,infra_evidence,Infrastructure: evidence / assessor notes,Briefly record the evidence supporting the score.,True
55,select_one score_1_4,facilities_score,"Facilities, equipment, furniture and teaching ...","Use 1=very poor, 2=poor, 3=good, 4=very good. ...",True
56,text,facilities_evidence,"Facilities, equipment, furniture and teaching ...",Briefly record the evidence supporting the score.,True
63,select_one score_1_4,learning_score,Graduate competency / learning outcomes: provi...,"Use 1=very poor, 2=poor, 3=good, 4=very good. ...",True
64,text,learning_evidence,Graduate competency / learning outcomes: evide...,Briefly record the evidence supporting the score.,True



CRITERION 1 START


,type,name,label::English (en)
33,begin_group,c1,1. Infrastructure



CRITERION 1 ROWS: 33 to 46


,type,name,label::English (en),hint::English (en),required,relevant
33,begin_group,c1,1. Infrastructure,NaN,False,NaN
34,integer,classrooms_total,Total classrooms,NaN,True,NaN
35,integer,classrooms_usable,Usable classrooms,NaN,True,NaN
36,select_one condition_4,roof_condition,Roof condition,NaN,True,NaN
37,select_one condition_4,floor_condition,Floor condition,NaN,True,NaN
38,select_one yes_no_unknown,electricity_available,Electricity available at the school?,NaN,True,NaN
39,select_one yes_no_unknown,internet_available,Usable internet/mobile data available at the s...,NaN,True,NaN
40,select_one yes_no_unknown,disability_access,Any disability-access feature available?,NaN,True,NaN
41,select_one yes_no_unknown,water_available,Usable water available at the school?,NaN,True,NaN
42,integer,toilets_functional,Number of functioning toilets,NaN,True,NaN


Notes:

Criterion 1 is Infrastructure and contains these required fields: total classrooms, usable classrooms, roof condition, floor condition, electricity, internet/mobile data, disability access, water, functioning toilets, handwashing, provisional 1–4 score, and evidence notes.

The assessor section also confirms `assessor_name`, `assessor_role`, and `assessor_role_other`

## STEP 6: Pull choice lists directly from the XLSForm
---
Need to confirm the choice values before building App v0.2 with:

```
School Identification
  - Province
  - Level
  - School
  - auto-filled IDs
Assessor Information
  - Assessor name
  - Assessor role
  - Other role
Criterion 1 — Infrastructure
  - all 10 underlying infrastructure questions
  - provisional score
  - required evidence note
```

And then after that we’ll add the Save Assessment button so one completed record actually writes to a CSV/dataframe.

In [35]:
def get_choice_list(list_name):
    """
    Return Gradio-friendly (label, value) choices
    from the XLSForm choices sheet.
    """
    subset = choices_df[
        choices_df["list_name"] == list_name
    ][
        ["name", "label::English (en)"]
    ].dropna()

    return list(
        zip(
            subset["label::English (en)"],
            subset["name"]
        )
    )


# Choice lists needed for assessor info + Criterion 1
assessor_role_choices = get_choice_list("assessor_role")
condition_choices = get_choice_list("condition_4")
yes_no_unknown_choices = get_choice_list("yes_no_unknown")
score_choices = get_choice_list("score_1_4")


print("Assessor roles:")
print(assessor_role_choices)

print("\nCondition choices:")
print(condition_choices)

print("\nYes / No / Unknown choices:")
print(yes_no_unknown_choices)

print("\nScore choices:")
print(score_choices)

Assessor roles:
[('Education officer', 'education_officer'), ('School manager / administrator', 'school_manager'), ('Salvation Army officer', 'salvation_army_officer'), ('External assessor', 'external_assessor'), ('Other', 'other')]

Condition choices:
[('Very poor / serious damage', 'very_poor'), ('Poor / substantial repair needed', 'poor'), ('Good / some repair or maintenance needed', 'good'), ('Very good / little or no repair needed', 'very_good')]

Yes / No / Unknown choices:
[('Yes', 'yes'), ('No', 'no'), ('Unknown / not confirmed', 'unknown')]

Score choices:
[('1 - Very poor', '1'), ('2 - Poor', '2'), ('3 - Good', '3'), ('4 - Very good', '4')]



<h3 align = "center">
    Milestone #2
</h3>

The next milestone is to put together one complete working screen:
---
School Identification → Assessor Information → Criterion 1: Infrastructure


## STEP 7: Build App v0.2

In [ ]:
# Helper function to show "Other assessor role" only when needed
def toggle_other_role(role):
    return gr.Textbox(
        visible=(role == "other"),
        interactive=True,
        value=""
    )


# ---------------------------------------------------------
# Build the app
# ---------------------------------------------------------

province_choices = sorted(
    schools_df["province_label"]
    .dropna()
    .unique()
    .tolist()
)

with gr.Blocks(title="DRC School Assessment") as demo_v02:

    gr.Markdown(
        """
        # DRC School Assessment
        **Pilot prototype**

        Complete the school identification and assessment information below.
        """
    )

    # =====================================================
    # SECTION 1: SCHOOL IDENTIFICATION
    # =====================================================

    gr.Markdown("## 1. School Identification")

    with gr.Row():

        province_dropdown = gr.Dropdown(
            choices=province_choices,
            label="Province",
            value=None
        )

        level_dropdown = gr.Dropdown(
            choices=[],
            label="School Level",
            value=None
        )

    school_dropdown = gr.Dropdown(
        choices=[],
        label="School",
        value=None,
        filterable=True
    )

    gr.Markdown("### School Details")

    with gr.Row():

        cona_box = gr.Textbox(
            label="CONA ID",
            interactive=False
        )

        secope_box = gr.Textbox(
            label="SECOPE ID",
            interactive=False
        )

    municipality_box = gr.Textbox(
        label="Municipality / Territory / Locality",
        interactive=False
    )

    education_level_box = gr.Textbox(
        label="Recorded Education Level",
        interactive=False
    )

    # Hidden internal key
    school_key_box = gr.Textbox(
        label="Internal School Key",
        visible=False
    )


    # =====================================================
    # SECTION 2: ASSESSOR INFORMATION
    # =====================================================

    gr.Markdown("## 2. Assessor Information")

    assessor_name = gr.Textbox(
        label="Assessor name",
        placeholder="Enter assessor name"
    )

    assessor_role = gr.Dropdown(
        choices=assessor_role_choices,
        label="Assessor role",
        value=None
    )

    assessor_role_other = gr.Textbox(
    label="Other assessor role",
    placeholder="Please specify your role",
    visible=False,
    interactive=True
    )


    # =====================================================
    # SECTION 3: CRITERION 1
    # =====================================================

    gr.Markdown(
        """
        ## 3. Criterion 1 — Infrastructure

        Record the observed school conditions before assigning the
        provisional infrastructure score.
        """
    )

    with gr.Row():

        classrooms_total = gr.Number(
            label="Total classrooms",
            precision=0,
            minimum=0
        )

        classrooms_usable = gr.Number(
            label="Usable classrooms",
            precision=0,
            minimum=0
        )

    with gr.Row():

        roof_condition = gr.Radio(
            choices=condition_choices,
            label="Roof condition"
        )

        floor_condition = gr.Radio(
            choices=condition_choices,
            label="Floor condition"
        )

    with gr.Row():

        electricity_available = gr.Radio(
            choices=yes_no_unknown_choices,
            label="Electricity available at the school?"
        )

        internet_available = gr.Radio(
            choices=yes_no_unknown_choices,
            label="Usable internet/mobile data available at the school?"
        )

    with gr.Row():

        disability_access = gr.Radio(
            choices=yes_no_unknown_choices,
            label="Any disability-access feature available?"
        )

        water_available = gr.Radio(
            choices=yes_no_unknown_choices,
            label="Usable water available at the school?"
        )

    with gr.Row():

        toilets_functional = gr.Number(
            label="Number of functioning toilets",
            precision=0,
            minimum=0
        )

        handwashing_functional = gr.Radio(
            choices=yes_no_unknown_choices,
            label="Functional handwashing facility available?"
        )


    # -----------------------------------------------------
    # PROVISIONAL SCORE
    # -----------------------------------------------------

    gr.Markdown("### Infrastructure Assessment")

    infra_score = gr.Radio(
        choices=score_choices,
        label="Infrastructure: provisional assessor score",
        info="1 = Very poor | 2 = Poor | 3 = Good | 4 = Very good"
    )

    infra_evidence = gr.Textbox(
        label="Infrastructure: evidence / assessor notes",
        placeholder="Briefly record the evidence supporting the score.",
        lines=4
    )


    # =====================================================
    # EVENTS
    # =====================================================

    # Province -> available levels
    province_dropdown.change(
        fn=get_levels,
        inputs=province_dropdown,
        outputs=level_dropdown
    )

    # Province + level -> available schools
    level_dropdown.change(
        fn=get_schools,
        inputs=[
            province_dropdown,
            level_dropdown
        ],
        outputs=school_dropdown
    )

    # School -> auto-filled school details
    school_dropdown.change(
        fn=get_school_details,
        inputs=school_dropdown,
        outputs=[
            school_key_box,
            cona_box,
            secope_box,
            municipality_box,
            education_level_box
        ]
    )

    # Assessor role -> show/hide "Other"
    assessor_role.change(
        fn=toggle_other_role,
        inputs=assessor_role,
        outputs=assessor_role_other
    )


# ---------------------------------------------------------
# Launch
# ---------------------------------------------------------

demo_v02.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f28e72818372933e9c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
